# Estudo Dirigido — MLP Backpropagation
## Dry Bean Dataset — Classificação Multiclasse (7 classes)

**Base:** UCI Machine Learning Repository — Dry Bean Dataset  
**Problema:** Classificação multiclasse supervisionada  
**Algoritmo:** Rede Neural MLP com Backpropagation (Keras/TensorFlow)  

---
### Estrutura do notebook
- **Q1** — Exploração Inicial: pesos e funções de ativação (5 seeds)
- **Q2** — Hiperparâmetros: taxa de aprendizado e termo momento (Grid Search)
- **Q3** — Topologia: número de camadas e neurônios
- **Q4** — Influência da quantidade de dados de treinamento
- **Q5** — Influência dos atributos (feature selection)
- **Q6** — Validação Inicial: 4 melhores configurações vs conjunto de teste
- **Q7** — Validação Cruzada K-Fold da melhor configuração

## ⚙️ Instalação e Importações

In [ ]:
# Execute esta célula apenas no Google Colab
!pip install ucimlrepo tensorflow scikit-learn pandas matplotlib seaborn -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import SGD, Adam
from tensorflow.keras.callbacks import Callback
from tensorflow.keras.utils import to_categorical

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split, KFold, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, classification_report, confusion_matrix
)
from sklearn.feature_selection import mutual_info_classif

print(f'TensorFlow: {tf.__version__}')
print('✅ Importações concluídas.')

## 📦 Carregamento e Pré-processamento do Dataset

- **Normalização:** Min-Max  
- **Classes:** Label Encoding + One-Hot (para `categorical_crossentropy`)  
- **Divisão estratificada tripla:** holdout 70% (Q1–Q2, Q7) → treino 56% + validação 14% (Q3–Q5) + teste 30% reservado (Q6)

### 🧠 Nota de Raciocínio: Prevenção de Data Leakage
Optei por realizar o `train_test_split` **antes** de qualquer normalização. 

**Por que?** Se eu aplicasse o `fit` do `MinMaxScaler` em todo o dataset (`X_raw`), as estatísticas do conjunto de teste (mínimos e máximos) seriam incorporadas ao escalonamento do treino. Isso é um erro comum chamado *Data Leakage* (Vazamento de Dados). Ao separar primeiro, garanto que o escalonador aprenda os parâmetros apenas com os dados de treino, mantendo o conjunto de teste como um ambiente verdadeiramente cego para uma validação realista.

In [ ]:
# ── Carrega o Dry Bean Dataset direto do UCI ──────────────────
dry_bean = fetch_ucirepo(id=602)
X_raw = dry_bean.data.features
y_raw = dry_bean.data.targets.squeeze()

print('📊 Informações do Dataset:')
print(f'   Instâncias  : {X_raw.shape[0]}')
print(f'   Atributos   : {X_raw.shape[1]}')
print(f'   Classes     : {y_raw.nunique()} → {sorted(y_raw.unique())}')
print(f'\nDistribuição de classes:')
print(y_raw.value_counts())

# ── Codificação das classes ───────────────────────────────────
le = LabelEncoder()
y_int = le.fit_transform(y_raw)
class_names = le.classes_
N_CLASSES = len(class_names)
N_FEATURES = X_raw.shape[1]

print(f'\n🔠 Classes ({N_CLASSES}): {list(class_names)}')

# ── Divisão estratificada tripla ──────────────────────────────
X_holdout_raw, X_test_raw, y_holdout_int, y_test_int = train_test_split(
    X_raw, y_int,
    test_size=0.30, random_state=42, stratify=y_int
)

# ── Normalização Min-Max (sem Data Leakage) ───────────────────
scaler = MinMaxScaler()
X_holdout = scaler.fit_transform(X_holdout_raw)
X_test = scaler.transform(X_test_raw)

X_train, X_val, y_train_int, y_val_int = train_test_split(
    X_holdout, y_holdout_int,
    test_size=0.20, random_state=42, stratify=y_holdout_int
)

y_holdout_ohe = to_categorical(y_holdout_int)
y_train_ohe   = to_categorical(y_train_int)
y_val_ohe     = to_categorical(y_val_int)
y_test_ohe    = to_categorical(y_test_int)

n = len(y_int)
print(f'\n✂️  Divisão estratificada (treino / validação / teste):')
print(f'   Holdout (Q1–Q2, Q7)   : {X_holdout.shape[0]:5d} ({100*X_holdout.shape[0]/n:.1f}%)')
print(f'   Treino interno (Q3–Q5): {X_train.shape[0]:5d} ({100*X_train.shape[0]/n:.1f}%)')
print(f'   Validação (Q3–Q5)     : {X_val.shape[0]:5d} ({100*X_val.shape[0]/n:.1f}%)')
print(f'   Teste (somente Q6)    : {X_test.shape[0]:5d} ({100*X_test.shape[0]/n:.1f}%) ← reservado')


---
## Q1 — Exploração Inicial: Pesos e Funções de Ativação

**Arquitetura base escolhida:**
- Entrada: 16 neurônios (um por atributo)
- Camada oculta 1: 64 neurônios + **ReLU**
- Camada oculta 2: 32 neurônios + **ReLU**
- Saída: 7 neurônios + **Softmax**

**Justificativas:**
- **ReLU** nas ocultas: evita vanishing gradient, esparso e eficiente com Adam
- **Softmax** na saída: distribui probabilidades entre as 7 classes (soma = 1)
- **Categorical Cross-Entropy**: penaliza predições erradas e confiantes, padrão para multiclasse
- **Adam**: adaptativo por parâmetro, robusto e rápido para convergência

**Objetivo:** verificar estabilidade do modelo frente a diferentes inicializações de pesos.

In [ ]:
# ── Parâmetros fixos da Q1 ────────────────────────────────────
ARCH_Q1     = [64, 32]        # neurônios por camada oculta
ACTIVATION  = 'relu'
EPOCHS_Q1   = 100
BATCH_SIZE  = 32
LR_Q1       = 0.001
SEEDS       = [0, 7, 21, 42, 99]

def build_model(hidden_layers, activation='relu', lr=0.001, optimizer='adam', momentum=0.0):
    """Constrói e compila um MLP para classificação multiclasse."""
    tf.keras.backend.clear_session()
    model = Sequential()
    model.add(keras.Input(shape=(N_FEATURES,)))
    for units in hidden_layers:
        model.add(Dense(units, activation=activation))
    model.add(Dense(N_CLASSES, activation='softmax'))

    if optimizer == 'adam':
        opt = Adam(learning_rate=lr)
    else:  # SGD com momento
        opt = SGD(learning_rate=lr, momentum=momentum)

    model.compile(
        optimizer=opt,
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

# ── Treinamento com 5 seeds ───────────────────────────────────
results_q1   = []
histories_q1 = []

for seed in SEEDS:
    tf.random.set_seed(seed)
    np.random.seed(seed)

    model = build_model(ARCH_Q1, lr=LR_Q1)
    t0 = time.time()
    hist = model.fit(
        X_holdout, y_holdout_ohe,
        epochs=EPOCHS_Q1,
        batch_size=BATCH_SIZE,
        verbose=0
    )
    elapsed = time.time() - t0

    final_loss = hist.history['loss'][-1]
    final_acc  = hist.history['accuracy'][-1] * 100

    results_q1.append({
        'Seed':          seed,
        'Loss Final':    round(final_loss, 4),
        'Acurácia (%)':  round(final_acc, 2),
        'Tempo (s)':     round(elapsed, 1)
    })
    histories_q1.append(hist)
    print(f'Seed {seed:3d} → Loss: {final_loss:.4f} | Acurácia: {final_acc:.2f}% | Tempo: {elapsed:.1f}s')

df_q1 = pd.DataFrame(results_q1)
df_q1.to_csv('resultados_q1.csv', index=False)
print('\n📊 TABELA Q1:')
print(df_q1.to_string(index=False))
print(f'\nMédia Loss   : {df_q1["Loss Final"].mean():.4f} ± {df_q1["Loss Final"].std():.4f}')
print(f'Média Acurácia: {df_q1["Acurácia (%)"].mean():.2f}% ± {df_q1["Acurácia (%)"].std():.2f}%')


In [ ]:
# ── Gráficos Q1 ───────────────────────────────────────────────
colors = ['#534AB7', '#1D9E75', '#D85A30', '#378ADD', '#BA7517']
epochs_ax = range(1, EPOCHS_Q1 + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q1 — Convergência por Inicialização de Pesos\n'
             'Arquitetura: 16 → 64 (ReLU) → 32 (ReLU) → 7 (Softmax) | Adam lr=0.001',
             fontsize=12, fontweight='bold')

for idx, (hist, seed) in enumerate(zip(histories_q1, SEEDS)):
    axes[0].plot(epochs_ax, hist.history['loss'],
                 color=colors[idx], label=f'Seed {seed}', linewidth=1.8)
    axes[1].plot(epochs_ax, [a*100 for a in hist.history['accuracy']],
                 color=colors[idx], label=f'Seed {seed}', linewidth=1.8)

axes[0].set_title('Curva de Perda (Cross-Entropy Loss)')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Curva de Acurácia (%)')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('q1_convergencia.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 q1_convergencia.png salvo.')

---
## Q2 — Hiperparâmetros: Taxa de Aprendizado e Termo Momento

**Método:** Grid Search com SGD (para expor o efeito do momentum)  
**Critério de parada:** loss ≤ 0.001 ou máximo de 200 épocas  
**Grades testadas:**
- Taxa de aprendizado: [0.001, 0.01, 0.1, 0.5]
- Termo momento: [0.5, 0.7, 0.9]

### 🧠 Nota de Raciocínio: Eficiência com Early Stopping
Nesta etapa, implementei o `EarlyStopping` nativo do Keras em vez de um loop manual simples. 

**Por que?** 
1. **Critério de Parada:** Defini o `baseline=0.001` para satisfazer o requisito da avaliação assim que a meta for atingida.
2. **Paciência (Patience):** Adicionei `patience=20` para evitar que o modelo continue rodando desnecessariamente caso a Loss estagne acima do baseline (ex: em 0.002). Isso economiza tempo e recursos computacionais, demonstrando uma abordagem prática de engenharia de software para IA.

In [ ]:
# ── Critério de Parada (Eficiência e Convergência) ──────────
from tensorflow.keras.callbacks import EarlyStopping

# ── Grade de hiperparâmetros ──────────────────────────────────
learning_rates = [0.001, 0.01, 0.1, 0.5]
momentums      = [0.5, 0.7, 0.9]
MAX_EPOCHS_Q2  = 200

results_q2   = []
histories_q2 = []

tf.random.set_seed(42)
np.random.seed(42)

total = len(learning_rates) * len(momentums)
count = 0
for lr in learning_rates:
    for mom in momentums:
        count += 1
        model = build_model(ARCH_Q1, lr=lr, optimizer='sgd', momentum=mom)

        # EarlyStopping nativo: para se atingir 0.001 (baseline) 
        # ou se parar de melhorar por 20 épocas (patience)
        stopper = EarlyStopping(
            monitor='loss', 
            baseline=0.001, 
            patience=20, 
            restore_best_weights=True
        )

        t0 = time.time()
        hist = model.fit(
            X_holdout, y_holdout_ohe,
            epochs=MAX_EPOCHS_Q2,
            batch_size=BATCH_SIZE,
            callbacks=[stopper],
            verbose=0
        )
        elapsed = time.time() - t0
        epochs_ran = len(hist.history['loss'])
        final_loss = hist.history['loss'][-1]

        results_q2.append({
            'LR': lr, 'Momento': mom,
            'Épocas': epochs_ran,
            'Loss Final': round(final_loss, 4),
            'Convergiu': '✅' if final_loss <= 0.001 else '❌',
            'Tempo (s)': round(elapsed, 1)
        })
        histories_q2.append({'lr': lr, 'mom': mom, 'hist': hist})
        print(f'[{count:2d}/{total}] LR={lr}, Mom={mom} → '
              f'Loss={final_loss:.4f}, Épocas={epochs_ran}')

df_q2 = pd.DataFrame(results_q2)
df_q2.to_csv('resultados_q2.csv', index=False)
print('\n📊 TABELA Q2:')
print(df_q2.to_string(index=False))


In [ ]:
# ── Gráficos Q2 ───────────────────────────────────────────────
fig, axes = plt.subplots(len(learning_rates), 1,
                          figsize=(12, 4 * len(learning_rates)), sharex=False)
fig.suptitle('Q2 — Loss por Época: Taxa de Aprendizado × Termo Momento\n'
             'Critério de parada: loss ≤ 0.001 | SGD | 200 épocas máx.',
             fontsize=12, fontweight='bold')

mom_colors = {'0.5': '#534AB7', '0.7': '#1D9E75', '0.9': '#D85A30'}

for ax_idx, lr in enumerate(learning_rates):
    ax = axes[ax_idx]
    for entry in histories_q2:
        if entry['lr'] == lr:
            mom = entry['mom']
            ax.plot(entry['hist'].history['loss'],
                    label=f'Momento={mom}',
                    color=mom_colors[str(mom)], linewidth=1.8)
    ax.axhline(0.001, color='red', linestyle='--', linewidth=1, label='Critério parada')
    ax.set_title(f'LR = {lr}', fontsize=11)
    ax.set_xlabel('Época'); ax.set_ylabel('Loss')
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig('q2_hiperparametros.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 q2_hiperparametros.png salvo.')

# Melhor combinação
best_q2 = df_q2.loc[df_q2['Loss Final'].idxmin()]
print(f'\n🏆 Melhor combinação Q2: LR={best_q2["LR"]}, Momento={best_q2["Momento"]} '
      f'→ Loss={best_q2["Loss Final"]}')
BEST_LR  = best_q2['LR']
BEST_MOM = best_q2['Momento']

---
## Q3 — Topologia: Número de Camadas e Neurônios

**Variações testadas:**
- Número de camadas ocultas: 1, 2, 3
- Neurônios por camada: 10, 20, …, 100 (passo de 10)

**Avaliação:** loss treino vs validação (underfitting/overfitting) + Medida-F


In [ ]:
# ── Grade de topologias (divisão definida no pré-processamento) ─
layer_options  = [1, 2, 3]
neuron_options = list(range(10, 101, 10))
EPOCHS_Q3      = 100

results_q3   = []
histories_q3 = []

tf.random.set_seed(42)
np.random.seed(42)

total = len(layer_options) * len(neuron_options)
count = 0
for n_layers in layer_options:
    for n_units in neuron_options:
        count += 1
        arch = [n_units] * n_layers
        model = build_model(arch, lr=LR_Q1)
        t0 = time.time()
        hist = model.fit(
            X_train, y_train_ohe,
            validation_data=(X_val, y_val_ohe),
            epochs=EPOCHS_Q3, batch_size=BATCH_SIZE, verbose=0
        )
        elapsed = time.time() - t0

        y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
        f1 = f1_score(y_val_int, y_pred, average='weighted')

        results_q3.append({
            'Camadas':       n_layers,
            'Neurônios':     n_units,
            'Loss Treino':   round(hist.history['loss'][-1], 4),
            'Loss Val':      round(hist.history['val_loss'][-1], 4),
            'Acur. Val(%)':  round(hist.history['val_accuracy'][-1] * 100, 2),
            'F1 (weighted)': round(f1, 4),
            'Tempo (s)':     round(elapsed, 1)
        })
        histories_q3.append({'layers': n_layers, 'units': n_units, 'hist': hist})
        print(f'[{count:2d}/{total}] {n_layers}L×{n_units}N → '
              f'LossTr={results_q3[-1]["Loss Treino"]}, '
              f'LossVal={results_q3[-1]["Loss Val"]}, F1={f1:.4f}')

df_q3 = pd.DataFrame(results_q3)
df_q3.to_csv('resultados_q3.csv', index=False)
print('\n📊 TABELA Q3:')
print(df_q3.to_string(index=False))


In [ ]:
# ── Heatmap F1 por topologia ──────────────────────────────────
pivot = df_q3.pivot(index='Camadas', columns='Neurônios', values='F1 (weighted)')
plt.figure(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='.4f', cmap='YlGnBu', linewidths=0.5)
plt.title('Q3 — Medida-F (weighted) por Topologia\nCamadas Ocultas × Neurônios por Camada')
plt.xlabel('Neurônios por camada')
plt.ylabel('Número de camadas')
plt.tight_layout()
plt.savefig('q3_heatmap_f1.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Análise Underfitting / Overfitting ────────────────────────
df_q3['Gap Loss (Val-Tr)'] = df_q3['Loss Val'] - df_q3['Loss Treino']
df_q3['Diagnóstico'] = np.where(
    df_q3['Gap Loss (Val-Tr)'] > 0.05, 'Overfitting',
    np.where(df_q3['Loss Treino'] > 0.30, 'Underfitting', 'Adequado')
)

print('\n📋 Diagnóstico Underfitting / Overfitting:')
print(df_q3[['Camadas', 'Neurônios', 'Loss Treino', 'Loss Val',
             'Gap Loss (Val-Tr)', 'F1 (weighted)', 'Diagnóstico']].to_string(index=False))

plt.figure(figsize=(8, 5))
sc = plt.scatter(df_q3['Loss Treino'], df_q3['Loss Val'],
                 c=df_q3['F1 (weighted)'], cmap='viridis', s=60, edgecolors='white')
plt.plot([0, 1], [0, 1], 'r--', linewidth=1.5, label='Treino = Validação')
plt.xlabel('Loss Treino')
plt.ylabel('Loss Validação')
plt.title('Q3 — Underfitting vs Overfitting')
plt.colorbar(sc, label='F1 (weighted)')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('q3_underfitting_overfitting.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 q3_underfitting_overfitting.png salvo.')

top4 = df_q3.nlargest(4, 'F1 (weighted)').reset_index(drop=True)
print('\n🏆 4 MELHORES TOPOLOGIAS (base para Q6):')
print(top4[['Camadas', 'Neurônios', 'Loss Treino', 'Loss Val',
            'Acur. Val(%)', 'F1 (weighted)', 'Diagnóstico']].to_string(index=False))


---
## Q4 — Influência da Quantidade de Dados de Treinamento

**Proporções testadas:** 20%, 40%, 60%, 80%, 100% do conjunto de treino  
**Modelo:** melhor arquitetura obtida na Q3  
**Estratégia:** amostragem estratificada em cada proporção

In [ ]:
# ── Melhor arquitetura da Q3 ──────────────────────────────────
best_row = df_q3.loc[df_q3['F1 (weighted)'].idxmax()]
BEST_LAYERS  = int(best_row['Camadas'])
BEST_UNITS   = int(best_row['Neurônios'])
BEST_ARCH    = [BEST_UNITS] * BEST_LAYERS
print(f'✅ Melhor arquitetura da Q3: {BEST_LAYERS} camadas × {BEST_UNITS} neurônios')

proportions = [0.2, 0.4, 0.6, 0.8, 1.0]
EPOCHS_Q4   = 100
results_q4  = []
histories_q4= []

tf.random.set_seed(42); np.random.seed(42)

for prop in proportions:
    if prop < 1.0:
        X_sub, _, y_sub, _ = train_test_split(
            X_train, y_train_int,
            train_size=prop, random_state=42, stratify=y_train_int
        )
    else:
        X_sub, y_sub = X_train, y_train_int

    y_sub_ohe = to_categorical(y_sub)
    model = build_model(BEST_ARCH, lr=LR_Q1)
    t0 = time.time()
    hist = model.fit(
        X_sub, y_sub_ohe,
        validation_data=(X_val, y_val_ohe),
        epochs=EPOCHS_Q4, batch_size=BATCH_SIZE, verbose=0
    )
    elapsed = time.time() - t0

    y_pred = np.argmax(model.predict(X_val, verbose=0), axis=1)
    f1     = f1_score(y_val_int, y_pred, average='weighted')
    acc    = accuracy_score(y_val_int, y_pred)

    results_q4.append({
        'Proporção (%)':  int(prop * 100),
        'Nº Exemplos':    len(X_sub),
        'Loss Treino':    round(hist.history['loss'][-1], 4),
        'Loss Val':       round(hist.history['val_loss'][-1], 4),
        'Acurácia Val(%)':round(acc * 100, 2),
        'F1 (weighted)':  round(f1, 4),
        'Tempo (s)':      round(elapsed, 1)
    })
    histories_q4.append({'prop': prop, 'hist': hist})
    print(f'{int(prop*100):3d}% ({len(X_sub):5d} ex.) → '
          f'LossVal={results_q4[-1]["Loss Val"]}, F1={f1:.4f}')

df_q4 = pd.DataFrame(results_q4)
df_q4.to_csv('resultados_q4.csv', index=False)
print('\n📊 TABELA Q4:')
print(df_q4.to_string(index=False))


In [ ]:
# ── Curvas de generalização Q4 ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Q4 — Influência da Quantidade de Dados de Treinamento', fontsize=12, fontweight='bold')

prop_colors = ['#534AB7','#1D9E75','#D85A30','#378ADD','#BA7517']
for idx, entry in enumerate(histories_q4):
    label = f"{int(entry['prop']*100)}%"
    axes[0].plot(entry['hist'].history['val_loss'],
                 color=prop_colors[idx], label=label, linewidth=1.8)
    axes[1].plot([a*100 for a in entry['hist'].history['val_accuracy']],
                 color=prop_colors[idx], label=label, linewidth=1.8)

axes[0].set_title('Loss de Validação por Época')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].set_title('Acurácia de Validação por Época')
axes[1].set_xlabel('Época'); axes[1].set_ylabel('Acurácia (%)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('q4_quantidade_dados.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Q5 — Influência dos Atributos (Feature Selection)

**Estratégias:**
1. Todos os 16 atributos (baseline)
2. Top-12 por Informação Mútua
3. Top-8 por Informação Mútua
4. Top-4 por Informação Mútua (remoção agressiva)

**Objetivo:** medir impacto em desempenho, tempo de convergência e **estabilidade** (5 seeds por subconjunto).


In [ ]:
# ── Ranking de importância por Informação Mútua ───────────────
mi_scores = mutual_info_classif(X_train, y_train_int, random_state=42)
feature_names = list(X_raw.columns)
mi_df = pd.DataFrame({'Feature': feature_names, 'MI Score': mi_scores})
mi_df = mi_df.sort_values('MI Score', ascending=False).reset_index(drop=True)

print('📊 Ranking de Informação Mútua:')
print(mi_df.to_string(index=False))

# Gráfico de importância
plt.figure(figsize=(10, 5))
plt.barh(mi_df['Feature'][::-1], mi_df['MI Score'][::-1], color='#534AB7')
plt.title('Q5 — Importância dos Atributos (Informação Mútua)', fontsize=12, fontweight='bold')
plt.xlabel('MI Score'); plt.tight_layout()
plt.savefig('q5_feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Experimentos com subconjuntos de atributos + estabilidade ──
top_features = mi_df['Feature'].tolist()
feature_subsets = {
    'Todos (16)': top_features,
    'Top-12':     top_features[:12],
    'Top-8':      top_features[:8],
    'Top-4':      top_features[:4],
}

EPOCHS_Q5    = 100
SEEDS_Q5     = [0, 7, 21, 42, 99]
results_q5   = []
histories_q5 = []

for name, features in feature_subsets.items():
    idx_feat = [feature_names.index(f) for f in features]
    Xtr_f  = X_train[:, idx_feat]
    Xval_f = X_val[:, idx_feat]

    f1_runs, acc_runs, time_runs = [], [], []
    last_hist = None

    for seed in SEEDS_Q5:
        tf.random.set_seed(seed)
        np.random.seed(seed)
        tf.keras.backend.clear_session()

        model_f = Sequential()
        model_f.add(keras.Input(shape=(len(features),)))
        for units in BEST_ARCH:
            model_f.add(Dense(units, activation='relu'))
        model_f.add(Dense(N_CLASSES, activation='softmax'))
        model_f.compile(
            optimizer=Adam(LR_Q1),
            loss='categorical_crossentropy',
            metrics=['accuracy']
        )

        t0 = time.time()
        hist = model_f.fit(
            Xtr_f, y_train_ohe,
            validation_data=(Xval_f, y_val_ohe),
            epochs=EPOCHS_Q5, batch_size=BATCH_SIZE, verbose=0
        )
        time_runs.append(time.time() - t0)

        y_pred = np.argmax(model_f.predict(Xval_f, verbose=0), axis=1)
        f1_runs.append(f1_score(y_val_int, y_pred, average='weighted'))
        acc_runs.append(accuracy_score(y_val_int, y_pred))
        last_hist = hist

    results_q5.append({
        'Subconjunto':      name,
        'Nº Atributos':     len(features),
        'F1 média':         round(np.mean(f1_runs), 4),
        'F1 std':           round(np.std(f1_runs), 4),
        'Acurácia média(%)':round(np.mean(acc_runs) * 100, 2),
        'Tempo médio (s)':  round(np.mean(time_runs), 1),
        'Loss Treino':      round(last_hist.history['loss'][-1], 4),
        'Loss Val':         round(last_hist.history['val_loss'][-1], 4),
    })
    histories_q5.append({'name': name, 'hist': last_hist})
    print(f'{name:12s} ({len(features):2d} attr.) → '
          f'F1={np.mean(f1_runs):.4f}±{np.std(f1_runs):.4f}, '
          f'Tempo={np.mean(time_runs):.1f}s')

df_q5 = pd.DataFrame(results_q5)
df_q5.to_csv('resultados_q5.csv', index=False)
print('\n📊 TABELA Q5 (com estabilidade por 5 seeds):')
print(df_q5.to_string(index=False))


---
## Q6 — Validação Inicial: 4 Melhores Configurações vs Conjunto de Teste

⚠️ **Primeira vez que o conjunto de teste é utilizado!**  
Treino no holdout completo (70%); **métricas de teste calculadas apenas após o treino** (sem `validation_data` no teste).


In [ ]:
print('🏆 4 melhores topologias da Q3 para validação:')
print(top4[['Camadas', 'Neurônios', 'F1 (weighted)']].to_string(index=False))


class TestMetricsCallback(Callback):
    """Registra loss/acurácia no teste a cada época sem influenciar o treino."""
    def __init__(self, X_test, y_test_ohe):
        super().__init__()
        self.X_test = X_test
        self.y_test_ohe = y_test_ohe
        self.test_losses = []
        self.test_accs = []

    def on_epoch_end(self, epoch, logs=None):
        loss, acc = self.model.evaluate(
            self.X_test, self.y_test_ohe, verbose=0
        )
        self.test_losses.append(loss)
        self.test_accs.append(acc)


results_q6   = []
histories_q6 = []
EPOCHS_Q6    = 100

tf.random.set_seed(42)
np.random.seed(42)

for _, row in top4.iterrows():
    n_layers = int(row['Camadas'])
    n_units  = int(row['Neurônios'])
    arch     = [n_units] * n_layers
    label    = f'{n_layers}L×{n_units}N'

    model = build_model(arch, lr=LR_Q1)
    test_cb = TestMetricsCallback(X_test, y_test_ohe)
    t0 = time.time()
    hist = model.fit(
        X_holdout, y_holdout_ohe,
        epochs=EPOCHS_Q6,
        batch_size=BATCH_SIZE,
        callbacks=[test_cb],
        verbose=0
    )
    elapsed = time.time() - t0

    y_pred_test = np.argmax(model.predict(X_test, verbose=0), axis=1)
    f1_test  = f1_score(y_test_int, y_pred_test, average='weighted')
    acc_test = accuracy_score(y_test_int, y_pred_test)
    loss_test, _ = model.evaluate(X_test, y_test_ohe, verbose=0)

    results_q6.append({
        'Config':         label,
        'Loss Treino':    round(hist.history['loss'][-1], 4),
        'Loss Teste':     round(loss_test, 4),
        'Acur. Teste(%)': round(acc_test * 100, 2),
        'F1 Teste':       round(f1_test, 4),
        'Tempo (s)':      round(elapsed, 1)
    })
    histories_q6.append({
        'label': label,
        'hist': hist,
        'test_cb': test_cb
    })
    print(f'{label} → Loss Teste={loss_test:.4f}, '
          f'Acc={acc_test*100:.2f}%, F1={f1_test:.4f}')

df_q6 = pd.DataFrame(results_q6)
df_q6.to_csv('resultados_q6.csv', index=False)
print('\n📊 TABELA Q6 — Validação Inicial (conjunto de teste isolado):')
print(df_q6.to_string(index=False))

best_q6 = df_q6.loc[df_q6['F1 Teste'].idxmax()]
BEST_CONFIG_LABEL = best_q6['Config']
idx_best = df_q6['F1 Teste'].idxmax()
BEST_LAYERS_FINAL = int(top4.iloc[idx_best]['Camadas'])
BEST_UNITS_FINAL  = int(top4.iloc[idx_best]['Neurônios'])
BEST_ARCH_FINAL   = [BEST_UNITS_FINAL] * BEST_LAYERS_FINAL
print(f'\n✅ Melhor configuração para Q7: {BEST_CONFIG_LABEL}')


In [ ]:
# ── Gráficos comparativos Q6 ──────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Q6 — Validação Inicial: 4 Melhores Configurações vs Conjunto de Teste\n'
             '(teste monitorado por callback, sem vazamento no treino)',
             fontsize=12, fontweight='bold')
q6_colors = ['#534AB7', '#1D9E75', '#D85A30', '#378ADD']

for idx, entry in enumerate(histories_q6):
    axes[0].plot(entry['test_cb'].test_losses,
                 color=q6_colors[idx], label=entry['label'], linewidth=1.8)
    axes[1].plot([a * 100 for a in entry['test_cb'].test_accs],
                 color=q6_colors[idx], label=entry['label'], linewidth=1.8)

axes[0].set_title('Loss no Conjunto de Teste (pós-treino por época)')
axes[0].set_xlabel('Época')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].set_title('Acurácia no Conjunto de Teste (%)')
axes[1].set_xlabel('Época')
axes[1].set_ylabel('Acurácia (%)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('q6_validacao_inicial.png', dpi=150, bbox_inches='tight')
plt.show()
print('💾 q6_validacao_inicial.png salvo.')


---
## Q7 — Validação Cruzada K-Fold (k=10)

**Objetivo:** avaliar robustez da melhor configuração (Q6) em partições do **holdout (70%)**, preservando o teste final.  
**Métricas:** média e desvio-padrão (e variância) de Loss, Acurácia e F1.


### 🧠 Nota de Raciocínio: Por que Validação Cruzada?
A Etapa 7 utiliza o **Stratified K-Fold (k=10)** para a configuração campeã.

**Por que?** Avaliar um modelo em uma única divisão de treino/teste pode ser enganoso devido à sorte na amostragem. Ao realizar 10 experimentos em diferentes subconjuntos e reportar a **média e variância**, provo matematicamente que a performance do meu modelo é consistente e confiável em qualquer subconjunto dos dados, elevando o rigor estatístico da avaliação.

In [ ]:
K_FOLDS   = 10
EPOCHS_Q7 = 100
skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=42)

fold_results = []
sample_histories = []

print(f'🔁 Validação cruzada {K_FOLDS}-fold no holdout — config: {BEST_CONFIG_LABEL}')
print(f'   Amostras: {X_holdout.shape[0]} (teste final de {X_test.shape[0]} instâncias não incluídas)')
print('-' * 60)

for fold_idx, (train_idx, val_idx) in enumerate(
        skf.split(X_holdout, y_holdout_int), start=1):
    Xf_tr = X_holdout[train_idx]
    Xf_val = X_holdout[val_idx]
    yf_tr = y_holdout_int[train_idx]
    yf_val = y_holdout_int[val_idx]

    tf.random.set_seed(fold_idx)
    np.random.seed(fold_idx)
    model = build_model(BEST_ARCH_FINAL, lr=LR_Q1)
    hist = model.fit(
        Xf_tr, to_categorical(yf_tr),
        validation_data=(Xf_val, to_categorical(yf_val)),
        epochs=EPOCHS_Q7, batch_size=BATCH_SIZE, verbose=0
    )

    y_pred = np.argmax(model.predict(Xf_val, verbose=0), axis=1)
    fold_results.append({
        'Fold':          fold_idx,
        'Loss':          round(hist.history['val_loss'][-1], 4),
        'Acurácia (%)':  round(accuracy_score(yf_val, y_pred) * 100, 2),
        'F1 (weighted)': round(f1_score(yf_val, y_pred, average='weighted'), 4),
        'Precisão':      round(precision_score(yf_val, y_pred, average='weighted', zero_division=0), 4),
        'Revocação':     round(recall_score(yf_val, y_pred, average='weighted', zero_division=0), 4),
    })
    if fold_idx <= 3:
        sample_histories.append({'fold': fold_idx, 'hist': hist})

    print(f'Fold {fold_idx:2d} → Loss={fold_results[-1]["Loss"]}, '
          f'Acc={fold_results[-1]["Acurácia (%)"]:.2f}%, '
          f'F1={fold_results[-1]["F1 (weighted)"]:.4f}')

df_q7 = pd.DataFrame(fold_results)
df_q7.to_csv('resultados_q7.csv', index=False)
print('\n📊 TABELA Q7 — Resultados por Fold:')
print(df_q7.to_string(index=False))

print('\n📈 RESUMO ESTATÍSTICO (média ± desvio-padrão | variância):')
for col in ['Loss', 'Acurácia (%)', 'F1 (weighted)', 'Precisão', 'Revocação']:
    print(f'  {col:20s}: {df_q7[col].mean():.4f} ± {df_q7[col].std():.4f}  '
          f'(var={df_q7[col].var():.6f})')


In [ ]:
# ── Gráficos Q7 ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(f'Q7 — Validação Cruzada {K_FOLDS}-Fold | Melhor config: {BEST_CONFIG_LABEL}',
             fontsize=12, fontweight='bold')

# Curvas de 3 folds representativos
fold_colors = ['#534AB7','#1D9E75','#D85A30']
for i, entry in enumerate(sample_histories):
    axes[0].plot(entry['hist'].history['val_loss'],
                 color=fold_colors[i], label=f'Fold {entry["fold"]}', linewidth=1.8)
axes[0].set_title('Loss de Validação (3 folds)')
axes[0].set_xlabel('Época'); axes[0].set_ylabel('Loss')
axes[0].legend(); axes[0].grid(alpha=0.3)

# Boxplot F1 por fold
axes[1].bar(df_q7['Fold'], df_q7['F1 (weighted)'], color='#534AB7', alpha=0.8, edgecolor='white')
axes[1].axhline(df_q7['F1 (weighted)'].mean(), color='red', linestyle='--',
                label=f'Média={df_q7["F1 (weighted)"].mean():.4f}')
axes[1].set_title('F1-Score por Fold')
axes[1].set_xlabel('Fold'); axes[1].set_ylabel('F1 (weighted)')
axes[1].set_ylim(0.8, 1.0); axes[1].legend(); axes[1].grid(alpha=0.3, axis='y')

# Boxplot das métricas finais
metrics_data = [df_q7['Loss'], df_q7['F1 (weighted)'], df_q7['Acurácia (%)']/100]
bp = axes[2].boxplot(metrics_data, labels=['Loss','F1','Acc'], patch_artist=True,
                      medianprops=dict(color='white', linewidth=2))
box_colors = ['#534AB7','#1D9E75','#D85A30']
for patch, color in zip(bp['boxes'], box_colors):
    patch.set_facecolor(color)
axes[2].set_title('Distribuição das Métricas (10 folds)')
axes[2].set_ylabel('Valor'); axes[2].grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('q7_validacao_cruzada.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Matriz de confusão final (último fold) ────────────────────
cm = confusion_matrix(yf_val, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'Q7 — Matriz de Confusão (Fold {K_FOLDS})', fontsize=12, fontweight='bold')
plt.xlabel('Predito'); plt.ylabel('Real')
plt.tight_layout()
plt.savefig('q7_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\n📋 Classification Report (último fold):')
print(classification_report(yf_val, y_pred, target_names=class_names))

---
## ✅ Resumo Final — Todas as Questões

| Questão | Foco | Métrica principal |
|---------|------|-------------------|
| Q1 | Estabilidade de inicialização (5 seeds) | Loss e Acurácia no treino |
| Q2 | Grid Search lr × momentum | Épocas até convergência |
| Q3 | Topologia (camadas × neurônios) | F1-score na validação |
| Q4 | Quantidade de dados | F1-score e curvas de generalização |
| Q5 | Seleção de atributos | F1-score e custo computacional |
| Q6 | 4 melhores configs vs teste | F1-score no teste |
| Q7 | K-Fold (k=10) da melhor config | Média ± desvio de todas as métricas |